In [1]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# --- Data ---
sentences = [
    "I love this course",
    "This is amazing",
    "I hate this topic",
    "This is boring"
]
labels = np.array([1, 1, 0, 0])  # 1 = positive, 0 = negative
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)
padded = pad_sequences(sequences, padding='post')
vocab_size = len(tokenizer.word_index) + 1
max_len = padded.shape[1]

# --- SIMPLE RNN WITHOUT EMBEDDING ---
model_no_embedding = Sequential([
    Input(shape=(max_len, 1)),
    SimpleRNN(10),
    Dense(1, activation='sigmoid')
])

model_no_embedding.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# RNN expects 3D input → reshape
padded_reshaped = padded.reshape((padded.shape[0], padded.shape[1], 1))

# --- Training ---
print("Training RNN WITHOUT Embedding")
model_no_embedding.fit(
    padded_reshaped,
    labels,
    epochs=50,  # increased epochs for better learning on small dataset
    verbose=0
)

# --- Evaluate on training data ---
loss, accuracy = model_no_embedding.evaluate(padded_reshaped, labels, verbose=0)
print(f"\nTraining Accuracy: {accuracy*100:.2f}%")

# --- Test with new input sentences ---
test_sentences = [
    "I love this topic",
    "This is terrible",
    "I hate this course"
]

# Convert to sequences & pad
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_padded = pad_sequences(test_seq, maxlen=max_len, padding='post')
test_padded_reshaped = test_padded.reshape((test_padded.shape[0], test_padded.shape[1], 1))

# Predict
predictions = model_no_embedding.predict(test_padded_reshaped)
for sentence, pred in zip(test_sentences, predictions):
    label = 1 if pred >= 0.5 else 0
    print(f"Sentence: '{sentence}' → Predicted: {label} (Probability: {pred[0]:.2f})")


Training RNN WITHOUT Embedding



Training Accuracy: 100.00%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


Sentence: 'I love this topic' → Predicted: 0 (Probability: 0.48)
Sentence: 'This is terrible' → Predicted: 1 (Probability: 0.64)
Sentence: 'I hate this course' → Predicted: 1 (Probability: 0.57)


In [2]:
# SIMPLE RNN WITH EMBEDDING

model_with_embedding = Sequential([
    Input(shape=(max_len,)),
    Embedding(input_dim=vocab_size, output_dim=8),
    SimpleRNN(10),
    Dense(1, activation='sigmoid')
])

model_with_embedding.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nTraining RNN WITH Embedding")
model_with_embedding.fit(
    padded,
    labels,
    epochs=20,
    verbose=0
)

# --- Evaluate on training data ---
loss_emb, accuracy_emb = model_with_embedding.evaluate(padded, labels, verbose=0)
print(f"Training Accuracy WITH Embedding: {accuracy_emb*100:.2f}%")

# --- Test with new input sentences ---
test_sentences = [
    "I love this topic",
    "This is terrible",
    "I hate this course"
]

# Convert to sequences & pad
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_padded = pad_sequences(test_seq, maxlen=max_len, padding='post')

# Predict
predictions_emb = model_with_embedding.predict(test_padded)
for sentence, pred in zip(test_sentences, predictions_emb):
    label = 1 if pred >= 0.5 else 0
    print(f"Sentence: '{sentence}' → Predicted: {label} (Probability: {pred[0]:.2f})")



Training RNN WITH Embedding


Training Accuracy WITH Embedding: 75.00%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step


Sentence: 'I love this topic' → Predicted: 0 (Probability: 0.50)
Sentence: 'This is terrible' → Predicted: 1 (Probability: 0.52)
Sentence: 'I hate this course' → Predicted: 0 (Probability: 0.47)
